# 02 — Feature Engineering
60+ features: correlations, distributions

In [ ]:
import sys, os

sys.path.insert(0, "..")
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from data.ingestion.rest_client import BitgetRESTClient
from data.ingestion.resampler import OHLCVResampler, Timeframe
from features.engine import FeatureEngine

%matplotlib inline
cfg = {
    "exchange": {
        "name": "bitget",
        "symbols": ["BTC/USDT"],
        "type": "spot",
        "rate_limit": {"max_requests_per_second": 10},
    },
    "data": {"validation": {"max_price_jump_pct": 30}},
    "features": {"max_window_bars": 500, "min_bars_required": 50},
    "strategies": {"mtf_macd_elder": {"macd": {"fast": 12, "slow": 26, "signal": 9}}},
}
client = BitgetRESTClient(cfg)
df = client.fetch_days(timeframe="1h", days=90)
d4 = OHLCVResampler.resample_bulk(df, Timeframe.H4)
dd = OHLCVResampler.resample_bulk(df, Timeframe.D1)
engine = FeatureEngine(cfg)
feats = engine.bulk_compute(df, d4, dd)
print(f"{len(feats.columns)} features x {len(feats)} bars")

In [ ]:
fc = feats.dropna()
corr = fc.corrwith(df["close"].loc[fc.index]).sort_values(key=abs, ascending=False)
top = corr.head(20)
fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#00C9A7" if v > 0 else "#FF6B6B" for v in top.values]
ax.barh(range(len(top)), top.values, color=colors)
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top.index, fontsize=9)
ax.set_title("Feature Correlation with Close")
ax.axvline(0, color="white", lw=0.5)
plt.tight_layout()
plt.show()

In [ ]:
key = [
    "returns",
    "atr_14",
    "rsi_14",
    "macd_hist",
    "adx_14",
    "bb_width",
    "volume_sma_ratio",
]
fig, ax = plt.subplots(2, 4, figsize=(14, 8))
for i, c in enumerate(key):
    ax[i // 4, i % 4].hist(fc[c].dropna(), bins=50, color="#58a6ff", alpha=0.7)
    ax[i // 4, i % 4].set_title(c)
plt.tight_layout()
plt.show()